# C-04 OpenAI SDK 호환 방식으로 Ollama 호출

이 노트북은 OpenAI SDK를 사용하되, 실제 호출 대상은 로컬 Ollama의 OpenAI-compatible endpoint로 바꾸는 방식입니다.

## OpenAI-compatible endpoint란

Ollama는 자체 API(`/api/chat`) 외에도 OpenAI SDK와 비슷한 형식의 `/v1/chat/completions` API를 제공합니다. 그래서 코드에서는 `OpenAI(...)` 클라이언트를 쓰지만 `base_url`을 `http://localhost:11434/v1`로 바꾸면 로컬 Ollama로 요청이 갑니다.

## 이 노트북에서 보는 포인트

- OpenAI SDK 호출 문법을 유지하면서 로컬 모델을 사용합니다.
- 나중에 OpenAI API, vLLM, LiteLLM, Ollama 같은 OpenAI 호환 서버를 바꿔 끼우기 쉽습니다.
- `response_format={"type": "json_object"}`로 JSON 응답을 유도합니다.
- 최종적으로 Pydantic으로 한 번 더 검증합니다.

## C-01과 차이

C-01은 Ollama native API를 직접 호출합니다. C-04는 OpenAI SDK 표준 모양을 사용합니다. 팀에 OpenAI SDK 사용 경험이 있거나, 모델 서버를 자주 바꿀 가능성이 있으면 C-04가 더 익숙하고 이식성이 좋을 수 있습니다.

## 주의할 점

OpenAI SDK를 쓴다고 해서 OpenAI 클라우드로 요청이 가는 것은 아닙니다. 이 노트북에서는 `base_url`이 로컬 Ollama를 가리키므로 온프레미스 전제를 유지합니다.

In [ ]:
import json
import os
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

try:
    from openai import OpenAI
except ImportError as exc:
    raise RuntimeError("이 노트북을 실행하려면 openai 패키지를 설치하세요") from exc

# OpenAI SDK를 쓰지만 base_url은 로컬 Ollama의 /v1 endpoint입니다.
# api_key는 Ollama 로컬 서버에서는 실제 인증용으로 쓰이지 않지만 SDK가 요구하므로 임의 값을 넣습니다.
OLLAMA_OPENAI_BASE_URL = os.getenv("OLLAMA_OPENAI_BASE_URL", "http://localhost:11434/v1")
MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:latest")
client = OpenAI(base_url=OLLAMA_OPENAI_BASE_URL, api_key="ollama")

In [ ]:
class EmailAnalysisResult(BaseModel):
    # OpenAI-compatible 방식에서도 결과 검증은 애플리케이션 코드에서 책임지는 편이 안전합니다.
    # predicted_email_intent 라벨은 임시 업무 분류 체계입니다. 실제 운영 전 반드시 사용자 검토가 필요합니다.
    # - inquiry: 견적/납기/제품 문의처럼 아직 발주가 확정되지 않은 요청
    # - order: 구매 발주서, PO, 주문 확정처럼 실제 주문 처리로 이어지는 요청
    # - service: 클레임, 고장, 누수, 긴급 지원처럼 서비스/AS 대응이 필요한 요청
    # - technical: 도면, 사양, 기술 검토, 호환성 확인처럼 엔지니어링 판단이 필요한 요청
    # - other: 위 기준으로 분류하기 어렵거나 추가 업무 유형 정의가 필요한 메일
    # TODO: 실제 고객 메일 샘플을 보고 라벨 이름, 개수, 정의를 확정해야 합니다.
    predicted_email_intent: Literal["inquiry", "order", "service", "technical", "other"]
    # predicted_email_importance 라벨도 임시 우선순위 체계입니다. SLA, 업무 프로세스, 사용자 화면 정책에 맞게 조정해야 합니다.
    # - low: 참고/일반 정보성으로 즉시 처리가 필요하지 않은 메일
    # - normal: 통상 처리 기한 안에 대응하면 되는 일반 업무 메일
    # - high: 납기, 견적 마감, 고객 영향 등으로 우선 확인이 필요한 메일
    # - urgent: 긴급 수리, 선박 운항 영향, 즉시 회신 요구처럼 지연 시 손실이 큰 메일
    # TODO: predicted_email_importance는 단순 감정/단어가 아니라 실제 처리 SLA와 연결해 정의해야 합니다.
    predicted_email_importance: Literal["low", "normal", "high", "urgent"]
    # extracted_key_information은 모델이 이메일에서 추출한 핵심 업무 정보입니다.
    # 예: 견적번호, PO 번호, 제품명, 수량, 납기일, 선박명 등.
    # TODO: 실제 필드 목록이 확정되면 dict가 아니라 별도 Pydantic 모델로 바꾸는 것이 좋습니다.
    extracted_key_information: dict = Field(default_factory=dict)
    # predicted_assignee_area는 실제 개인 담당자라기보다 임시 담당 영역/팀 후보입니다.
    # 예: sales_team, service_team, technical_team, order_management 등.
    # TODO: 실제 사용자/팀/라우팅 규칙이 정리되면 assignee_user_id, assignee_team_id와 분리할지 결정해야 합니다.
    predicted_assignee_area: str
    # prediction_reasoning은 위 예측값을 낸 근거 설명입니다.
    # TODO: 실제 UI에 노출할지, 내부 감사/디버깅 용도로만 저장할지 결정해야 합니다.
    prediction_reasoning: str
    # predicted_needs_human_review는 AI가 사람 검토 필요성을 예측한 값입니다.
    # 최종 검토 상태가 아니며, 서비스 계층에서 정책/신뢰도/오류 여부와 함께 확정해야 합니다.
    predicted_needs_human_review: bool


# 발주 접수 확인 상황을 가정한 한글 샘플입니다.
sample_email = {
    "subject": "구매 발주서 접수 확인",
    "sender": "buyer@example.com",
    "body": "첨부한 선박 예비품 구매 발주서 기준으로 진행 부탁드립니다. 접수 여부와 예상 납기를 회신해 주세요.",
    "attachments": ["구매발주서_PO-2026-071.pdf"],
}

In [ ]:
# Pydantic 스키마를 프롬프트에 포함해 모델에게 원하는 JSON 구조를 알려줍니다.
prompt = f"""
선박 부품 제조사 업무 이메일을 분석하세요.
다음 구조에 맞는 JSON만 반환하세요:
{EmailAnalysisResult.model_json_schema()}

이메일:
{json.dumps(sample_email, ensure_ascii=False, indent=2)}
"""

# OpenAI SDK의 chat.completions.create 문법을 그대로 사용합니다.
# base_url만 Ollama로 바꿨기 때문에 요청은 로컬 모델로 갑니다.
completion = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    response_format={"type": "json_object"},
)
content = completion.choices[0].message.content
print(content)

In [ ]:
# JSON 모드가 있어도 스키마 필드가 모두 맞는지는 별도로 확인합니다.
try:
    result = EmailAnalysisResult.model_validate_json(content)
    print(result.model_dump_json(indent=2))
except ValidationError as exc:
    print(exc)